In [1]:
import os
# MLflow ve ilişkili kütüphanelerin C/Cython derlemelerini bypass etmeye zorlar
os.environ["MLFLOW_SKINNY"] = "true"


In [2]:
%pwd

'/mnt/c/Users/user/Desktop/Data Science Projects/End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/mnt/c/Users/user/Desktop/Data Science Projects/End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment'

In [5]:
import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

/mnt/c/Users/user/Desktop/Data Science Projects/End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import tensorflow as tf

I0000 00:00:1792566668.156495   19719 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792566668.177030   19719 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792566668.663224   19719 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792566687.252315   19719 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [7]:
import os
# Python'ı kesin olarak projenin ana klasörünün içine sokuyoruz
os.chdir("/mnt/c/Users/user/Desktop/Data Science Projects/End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment")
print("Yeni Konumunuz:", os.getcwd())


Yeni Konumunuz: /mnt/c/Users/user/Desktop/Data Science Projects/End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment


In [8]:
model = tf.keras.models.load_model("artifacts/training/model.keras")

E0000 00:00:1792566704.315002   19719 cuda_executor.cc:1690] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:68

W0000 00:00:1792566704.315412   20004 cuda_executor.cc:1710] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:77
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:107
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:112

W0000 00:00:1792566704.339877   19719 dlopen_checker.cc:38] Could not load dynamic library 'libcudart.so.12'; dlerror: libcudart.so.12: cannot open shared object file: No such file or directory
W0000 00:00:1792566704.339898   1971

In [9]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [10]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [11]:
# Sabit dosya yollarini Pylance'in gorebilmesi icin hucre icinde acikliyoruz
CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):
            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)
            create_directories([self.config.artifacts_root])

    def get_evaluation_config(self)  -> EvaluationConfig:
        eval_config = EvaluationConfig(
             path_of_model="artifacts/training/model.keras",
             training_data="artifacts/data_ingestion/",
             mlflow_uri="",
             all_params=self.params,
             params_image_size=self.params.IMAGE_SIZE,
             params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [12]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [ ]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size =self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self._valid_generator = valid_datagenerator.flow_from_directory(
            directory = self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self._valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    def log_into_mlflow(self):
        import mlflow
        import os
        from cnnClassifier import logger
        
        logger.info("Savunma sanayisi modunda yerel izole kayit baslatiliyor...")
        
        # 1. Adım: Hafızada kalmış olabilecek tüm DagsHub/Bulut çevre değişkenlerini tamamen siliyoruz
        for key in list(os.environ.keys()):
            if "MLFLOW" in key or "DAGSHUB" in key:
                del os.environ[key]
                
        # 2. Adım: MLflow'un otomatik ürettiği ve sunucuları bozan tüm gizli sistem etiketlerini kapatıyoruz
        os.environ["MLFLOW_TRACKING_DISABLE_SYSTEM_TAGS"] = "true"
        os.environ["MLFLOW_DISABLE_SYSTEM_METRICS_LOGGING"] = "true"
        
        # 3. Adım: Takip mekanizmasını projenin artifacts klasörü altındaki SQLite dosyasına kesin olarak kilitliyoruz
        mlflow.set_tracking_uri("sqlite:///guvenli_takip.db")
        
        # 4. Adım: tags={} vererek tüm arka plan kalıntılarını temizleyip run başlatıyoruz
        with mlflow.start_run(run_name="kidney_disease_secure_run", nested=False, tags={}):
            # Parametreleri kaydet
            mlflow.log_params(self.config.all_params)
            
            # Skor listesindeki loss ve accuracy değerlerini kaydet
            mlflow.log_metric("eval_loss", float(self.score[0]))
            mlflow.log_metric("eval_accuracy", float(self.score[1]))
            
            # Modeli artefakt olarak kaydet
            mlflow.keras.log_model(self.model, "model")
            logger.info("Tum metrikler yerel izole veritabanina basariyla islendi!")
    

In [16]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
    raise e

[2026-10-21 10:16:36,094: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-10-21 10:16:36,101: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-21 10:16:36,105: INFO: common: created directory at: artifacts]
Found 2204 images belonging to 2 classes.
138/138 ━━━━━━━━━━━━━━━━━━━━ 107s 773ms/step - accuracy: 0.7967 - loss: 0.5793
[2026-10-21 10:18:23,694: INFO: common: json file saved at: scores.json]
[2026-10-21 10:18:23,696: INFO: 2116935175: Savunma sanayisi modunda yerel izole kayit baslatiliyor...]


2026/10/21 10:18:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/21 10:18:24 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


[2026-10-21 10:19:42,900: INFO: 2116935175: Tum metrikler yerel izole veritabanina basariyla islendi!]
